# 03 — Modeling K-Means Clustering

**Proyek:** Segmentasi UMKM Kota Bandung menggunakan K-Means Clustering

**Tujuan Notebook Ini:**
- Feature selection dan scaling
- Menentukan jumlah cluster optimal (Elbow Method)
- Training model K-Means (K=3)
- Reduksi dimensi PCA 2D untuk visualisasi cluster
- Evaluasi metrik: Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index

> **Input:** `data/processed/03_Data_Modeling_Setelah_NLP.csv` (output dari notebook preprocessing)
>
> **Output:** `data/processed/04_Hasil_Clustering_Final.csv`

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

warnings.filterwarnings('ignore')

PROCESSED_PATH = os.path.join('..', 'data', 'processed')
print('Library berhasil dimuat.')

## 2. Memuat Data Hasil Preprocessing

In [ ]:
df = pd.read_csv(
    os.path.join(PROCESSED_PATH, '03_Data_Modeling_Setelah_NLP.csv'),
    sep=';', encoding='utf-8-sig'
)
print(f'Shape data: {df.shape}')
print(f'Kolom: {list(df.columns)}')
df.head()

## 3. Feature Selection & Scaling

Fitur yang digunakan untuk clustering:
1. `totalScore` — Rating Google Maps
2. `log_reviewsCount` — Log transformasi jumlah ulasan (mengurangi skewness)
3. `sentiment_score` — Skor sentimen NLP dari teks ulasan

In [ ]:
# Konversi tipe data
df['totalScore'] = pd.to_numeric(df['totalScore'].astype(str).str.replace(',', '.'), errors='coerce')
med_score = df['totalScore'].median()
df['totalScore'] = df['totalScore'].fillna(med_score if pd.notna(med_score) else 4.0)

df['reviewsCount'] = pd.to_numeric(df['reviewsCount'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)
df['log_reviewsCount'] = np.log1p(df['reviewsCount'])

df['sentiment_score'] = pd.to_numeric(df['sentiment_score'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0.5)

# Fitur untuk clustering
feature_cols = ['totalScore', 'log_reviewsCount', 'sentiment_score']
X = df[feature_cols].copy()

print('Fitur untuk clustering:')
print(X.describe())

# Scaling dengan StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nShape fitur setelah scaling: {X_scaled.shape}')

## 4. Elbow Method — Menentukan K Optimal

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('Jumlah Cluster (K)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K=3')
axes[0].legend()

# Silhouette
axes[1].plot(K_range, sil_scores, 'go-', linewidth=2)
axes[1].set_xlabel('Jumlah Cluster (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score per K')
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K=3')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nSilhouette Score per K:')
for k, s in zip(K_range, sil_scores):
    marker = ' <<<' if k == 3 else ''
    print(f'  K={k}: {s:.4f}{marker}')

## 5. Training K-Means (K=3)

In [ ]:
N_CLUSTERS = 3
RANDOM_STATE = 42

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Nama deskriptif per cluster (sesuai hasil penelitian)
cluster_names = {
    0: 'Cluster 1: Reputasi Positif \u2013 Volume Ulasan Rendah',
    1: 'Cluster 2: Reputasi Digital Perlu Perbaikan',
    2: 'Cluster 3: Performa Digital Tinggi'
}
df['cluster_name'] = df['cluster'].map(cluster_names)

# Distribusi cluster
print('=== DISTRIBUSI CLUSTER ===')
for c_name, count in df['cluster_name'].value_counts().items():
    pct = (count / len(df)) * 100
    print(f'  {c_name}: {count:,} UMKM ({pct:.1f}%)')

## 6. Reduksi Dimensi PCA 2D

PCA (Principal Component Analysis) 2D digunakan untuk memproyeksikan 3 fitur (`totalScore`, `log_reviewsCount`, `sentiment_score`) ke 2 dimensi agar dapat divisualisasikan.

> **Catatan:** Clustering tetap dilakukan pada 3 fitur asli. PCA 2D hanya untuk keperluan visualisasi hasil cluster.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coords = pca.fit_transform(X_scaled)
df['pca_x'] = pca_coords[:, 0]
df['pca_y'] = pca_coords[:, 1]

print(f'Explained Variance Ratio: {pca.explained_variance_ratio_}')
print(f'Total Variance Explained: {sum(pca.explained_variance_ratio_):.4f} ({sum(pca.explained_variance_ratio_)*100:.1f}%)')

## 7. Visualisasi Cluster PCA 2D

> Visualisasi ini adalah bagian dari **evaluasi model**, bukan EDA. Scatter plot PCA menunjukkan seberapa baik cluster terpisah satu sama lain.

In [ ]:
colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(12, 8))

for i in range(N_CLUSTERS):
    mask = df['cluster'] == i
    ax.scatter(
        df.loc[mask, 'pca_x'], df.loc[mask, 'pca_y'],
        c=colors[i], label=cluster_names[i],
        alpha=0.6, s=30, edgecolors='black', linewidth=0.3
    )

# Plot centroids
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=200, linewidth=2, label='Centroid'
)

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('Visualisasi Cluster K-Means (PCA 2D)')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## 8. Evaluasi Metrik Clustering

In [ ]:
sil = silhouette_score(X_scaled, df['cluster'])
dbi = davies_bouldin_score(X_scaled, df['cluster'])
chi = calinski_harabasz_score(X_scaled, df['cluster'])

print('=== METRIK EVALUASI CLUSTERING ===')
print(f'  Silhouette Score      : {sil:.4f}  (mendekati 1 = sangat baik)')
print(f'  Davies-Bouldin Index  : {dbi:.4f}  (semakin kecil = semakin baik)')
print(f'  Calinski-Harabasz     : {chi:.4f}  (semakin tinggi = semakin baik)')

## 9. Profil Statistik per Cluster

In [ ]:
profile = df.groupby('cluster_name')[feature_cols + ['reviewsCount']].agg(['mean', 'median', 'std', 'count'])
print('=== PROFIL STATISTIK PER CLUSTER ===')
profile

## 10. Simpan Hasil Clustering

In [ ]:
out_file = '04_Hasil_Clustering_Final.csv'
df.to_csv(
    os.path.join(PROCESSED_PATH, out_file),
    index=False, sep=';', encoding='utf-8-sig'
)
print(f'Hasil clustering tersimpan: {out_file}')
print(f'Shape: {df.shape}')

print('\n=== Langkah Selanjutnya ===')
print('Lanjutkan ke notebook 04_Rekomendasi_Bisnis.ipynb untuk rekomendasi per cluster.')